In [3]:
# From Underfitting to Overfitting: A Comparative Study of Linear and Polynomial Regression Models
# GitHub repository:
# https://github.com/hninhninphyu8895/FROM-UNDERFITTING-TO-OVERFITTING-A-COMPARATIVE-STUDY-OF-LINEAR-AND-POLYNOMIAL-REGRESSION-MODELS-.git

from __future__ import annotations

import io
import textwrap
import zipfile
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures

INDICATOR = "EN.GHG.CO2.PC.CE.AR5"
COUNTRY_CODE = "GBR"          
COUNTRY_NAME = "United Kingdom"
START_YEAR = 1970
MIN_NON_NULL_POINTS = 20
TEST_RATIO = 0.20
DEGREES = [1, 2, 3, 4, 5, 6, 8, 10, 12]
HIGH_DEGREE_FOR_RIDGE = 12
RIDGE_ALPHA = 10.0
RANDOM_SEED = 42
OUTPUT_DIR = Path("outputs")


COLORS: Dict[str, str] = {
    "navy": "#1f3b73",
    "sky": "#4ea8de",
    "teal": "#2a9d8f",
    "mint": "#80ed99",
    "gold": "#f4a261",
    "orange": "#e76f51",
    "rose": "#d0006f",
    "purple": "#6a4c93",
    "grey": "#6c757d",
    "light": "#f8f9fa",
}

plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "axes.titlesize": 13,
    "legend.frameon": False,
    "font.size": 10,
})


@dataclass
class ModelResult:
    degree: int
    model_name: str
    train_rmse: float
    test_rmse: float
    train_r2: float
    test_r2: float
    model: Pipeline


def download_world_bank_indicator(indicator: str) -> pd.DataFrame:
    url = f"https://api.worldbank.org/v2/en/indicator/{indicator}?downloadformat=csv"
    print(f"Downloading data from: {url}")

    response = requests.get(url, timeout=60)
    response.raise_for_status()

    with zipfile.ZipFile(io.BytesIO(response.content)) as zf:
        candidate_files = [
            name for name in zf.namelist()
            if name.endswith(".csv") and "Metadata" not in name
        ]
        if not candidate_files:
            raise FileNotFoundError("No data CSV found inside the downloaded ZIP archive.")

        candidate_files.sort(key=lambda x: (0 if "API_" in Path(x).name else 1, x))
        data_file = candidate_files[0]
        print(f"Reading CSV from archive: {data_file}")

        with zf.open(data_file) as f:
            df = pd.read_csv(f, skiprows=4)

    return df


def prepare_country_time_series(
    df: pd.DataFrame,
    country_code: str,
    country_name: str,
    start_year: int
) -> pd.DataFrame:
 
    
    if "Country Code" not in df.columns:
        raise KeyError("Expected 'Country Code' column not found in dataset.")

    country_df = df[df["Country Code"] == country_code].copy()

    if country_df.empty:
        if "Country Name" not in df.columns:
            raise KeyError("Expected 'Country Name' column not found in dataset.")
        matched = df[df["Country Name"].str.lower() == country_name.lower()].copy()
        if matched.empty:
            raise ValueError(f"Country '{country_code}' / '{country_name}' not found in dataset.")
        country_df = matched

    row = country_df.iloc[0]
    year_cols = [col for col in df.columns if str(col).isdigit()]

    ts = pd.DataFrame({
        "year": [int(y) for y in year_cols],
        "value": [pd.to_numeric(row[str(y)], errors="coerce") for y in year_cols],
    })

    ts = ts[ts["year"] >= start_year].dropna().reset_index(drop=True)

    if len(ts) < MIN_NON_NULL_POINTS:
        raise ValueError(
            f"Country '{country_code}' has only {len(ts)} valid observations from "
            f"{start_year} onward. Choose another country or lower START_YEAR."
        )

    ts["country_code"] = country_code
    ts["country_name"] = row["Country Name"]
    return ts


def chronological_split(ts: pd.DataFrame, test_ratio: float):
    n_test = max(3, int(np.ceil(len(ts) * test_ratio)))
    split_idx = len(ts) - n_test
    train_df = ts.iloc[:split_idx].copy()
    test_df = ts.iloc[split_idx:].copy()
    return train_df, test_df


def build_polynomial_model(degree: int, ridge_alpha: float | None = None) -> Pipeline:

    steps: List[tuple] = [
        ("poly", PolynomialFeatures(degree=degree, include_bias=False))
    ]

    if ridge_alpha is None:
        steps.append(("regressor", LinearRegression()))
    else:
        steps.append(("regressor", Ridge(alpha=ridge_alpha, random_state=RANDOM_SEED)))

    return Pipeline(steps)


def evaluate_model(
    model: Pipeline,
    X_train,
    y_train,
    X_test,
    y_test,
    degree: int,
    model_name: str
) -> ModelResult:

    model.fit(X_train, y_train)
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    return ModelResult(
        degree=degree,
        model_name=model_name,
        train_rmse=float(np.sqrt(mean_squared_error(y_train, train_pred))),
        test_rmse=float(np.sqrt(mean_squared_error(y_test, test_pred))),
        train_r2=float(r2_score(y_train, train_pred)),
        test_r2=float(r2_score(y_test, test_pred)),
        model=model,
    )


def make_prediction_grid(ts: pd.DataFrame, n_points: int = 500) -> np.ndarray:

    years = np.linspace(ts["year"].min(), ts["year"].max(), n_points)
    return years.reshape(-1, 1)


def add_caption(ax, text: str) -> None:
    ax.text(
        0.5,
        -0.22,
        textwrap.fill(text, width=100),
        transform=ax.transAxes,
        ha="center",
        va="top",
        fontsize=9,
        color=COLORS["grey"],
    )


def plot_raw_series(ts: pd.DataFrame, country_label: str, outpath: Path) -> None:

    fig, ax = plt.subplots(figsize=(11, 6))
    ax.plot(
        ts["year"],
        ts["value"],
        color=COLORS["navy"],
        linewidth=2.6,
        marker="o",
        markersize=4
    )
    ax.fill_between(ts["year"], ts["value"], color=COLORS["sky"], alpha=0.15)

    ax.set_title(f"Raw CO₂ Emissions Time Series: {country_label}")
    ax.set_xlabel("Year")
    ax.set_ylabel("CO₂ emissions excluding LULUCF per capita")
    ax.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(outpath, format="jpg", bbox_inches="tight")
    plt.close(fig)


def plot_model_fits(
    ts: pd.DataFrame,
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    results: List[ModelResult],
    outpath: Path
) -> None:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=True, sharey=True)
    years_grid = make_prediction_grid(ts)

    show_degrees = [1, 3, 8, 12]
    color_map = {
        1: COLORS["navy"],
        3: COLORS["teal"],
        8: COLORS["gold"],
        12: COLORS["rose"],
    }

    for ax, degree in zip(axes.ravel(), show_degrees):
        if degree == 1:
            result = next(r for r in results if r.degree == degree and r.model_name == "Linear")
        else:
            result = next(r for r in results if r.degree == degree and r.model_name == "Polynomial")

        preds = result.model.predict(years_grid)

        ax.scatter(
            train_df["year"], train_df["value"],
            color=COLORS["sky"], label="Train", s=34, alpha=0.9
        )
        ax.scatter(
            test_df["year"], test_df["value"],
            color=COLORS["orange"], label="Test", s=40, alpha=0.9
        )
        ax.plot(years_grid.ravel(), preds, color=color_map[degree], linewidth=2.8)

        ax.set_title(f"Degree {degree} | Test RMSE = {result.test_rmse:.3f}")
        ax.grid(alpha=0.2)

        if degree == 1:
            ax.text(
                0.03, 0.92, "Likely underfit",
                transform=ax.transAxes, fontsize=10,
                color=COLORS["navy"], weight="bold"
            )
        elif degree in (3, 8):
            ax.text(
                0.03, 0.92, "Stronger fit",
                transform=ax.transAxes, fontsize=10,
                color=COLORS["teal"], weight="bold"
            )
        else:
            ax.text(
                0.03, 0.92, "High flexibility",
                transform=ax.transAxes, fontsize=10,
                color=COLORS["rose"], weight="bold"
            )

    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=2, bbox_to_anchor=(0.5, 1.02))
    fig.supxlabel("Year")
    fig.supylabel("CO₂ emissions excluding LULUCF per capita")
    fig.suptitle(
        f"Linear and Polynomial Fits Across Increasing Model Complexity: {COUNTRY_NAME}",
        y=1.04,
        fontsize=15,
        fontweight="bold"
    )

    fig.tight_layout()
    fig.savefig(outpath, format="jpg", bbox_inches="tight")
    plt.close(fig)


def plot_error_curve(results: List[ModelResult], outpath: Path) -> None:
    poly_results = [r for r in results if r.model_name in ("Linear", "Polynomial")]
    poly_results = sorted(poly_results, key=lambda r: r.degree)

    degrees = [r.degree for r in poly_results]
    train_rmse = [r.train_rmse for r in poly_results]
    test_rmse = [r.test_rmse for r in poly_results]

    fig, ax = plt.subplots(figsize=(11, 6))
    ax.plot(degrees, train_rmse, marker="o", linewidth=2.5, color=COLORS["teal"], label="Train RMSE")
    ax.plot(degrees, test_rmse, marker="o", linewidth=2.5, color=COLORS["rose"], label="Test RMSE")

    best_degree = degrees[int(np.argmin(test_rmse))]
    best_test = min(test_rmse)

    ax.axvline(best_degree, color=COLORS["gold"], linestyle="--", linewidth=2, label=f"Best degree = {best_degree}")
    ax.scatter([best_degree], [best_test], color=COLORS["orange"], s=90, zorder=5)

    ax.set_title("Training vs Test Error Across Model Complexity")
    ax.set_xlabel("Polynomial degree")
    ax.set_ylabel("RMSE")
    ax.grid(alpha=0.25)
    ax.legend()

    fig.tight_layout()
    fig.savefig(outpath, format="jpg", bbox_inches="tight")
    plt.close(fig)


def plot_linear_vs_best(
    ts: pd.DataFrame,
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    linear_result: ModelResult,
    best_poly_result: ModelResult,
    outpath: Path
) -> None:

    fig, ax = plt.subplots(figsize=(11, 6))
    years_grid = make_prediction_grid(ts)

    ax.scatter(train_df["year"], train_df["value"], color=COLORS["sky"], s=38, label="Train")
    ax.scatter(test_df["year"], test_df["value"], color=COLORS["orange"], s=44, label="Test")

    ax.plot(
        years_grid.ravel(),
        linear_result.model.predict(years_grid),
        color=COLORS["navy"],
        linewidth=2.7,
        label=f"Linear (deg 1), test RMSE={linear_result.test_rmse:.3f}"
    )

    ax.plot(
        years_grid.ravel(),
        best_poly_result.model.predict(years_grid),
        color=COLORS["rose"],
        linewidth=2.7,
        label=f"Best polynomial (deg {best_poly_result.degree}), test RMSE={best_poly_result.test_rmse:.3f}"
    )

    ax.set_title("Linear vs Best Polynomial Regression")
    ax.set_xlabel("Year")
    ax.set_ylabel("CO₂ emissions excluding LULUCF per capita")
    ax.grid(alpha=0.25)
    ax.legend()
    fig.tight_layout()
    fig.savefig(outpath, format="jpg", bbox_inches="tight")
    plt.close(fig)


def plot_regularization_extension(
    ts: pd.DataFrame,
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    unregularized: ModelResult,
    regularized: ModelResult,
    outpath: Path
) -> None:

    fig, ax = plt.subplots(figsize=(11, 6))
    years_grid = make_prediction_grid(ts)

    ax.scatter(train_df["year"], train_df["value"], color=COLORS["sky"], s=35, label="Train")
    ax.scatter(test_df["year"], test_df["value"], color=COLORS["orange"], s=40, label="Test")

    ax.plot(
        years_grid.ravel(),
        unregularized.model.predict(years_grid),
        color=COLORS["rose"],
        linewidth=2.5,
        label=f"Degree {unregularized.degree} polynomial"
    )
    ax.plot(
        years_grid.ravel(),
        regularized.model.predict(years_grid),
        color=COLORS["purple"],
        linewidth=2.8,
        label=f"Degree {regularized.degree} + Ridge(alpha={RIDGE_ALPHA})"
    )

    ax.set_title("Optional Extension: Regularization of a High-Degree Polynomial")
    ax.set_xlabel("Year")
    ax.set_ylabel("CO₂ emissions excluding LULUCF per capita")
    ax.grid(alpha=0.25)
    ax.legend()

    fig.tight_layout()
    fig.savefig(outpath, format="jpg", bbox_inches="tight")
    plt.close(fig)


def save_metrics(results: List[ModelResult], outpath: Path) -> pd.DataFrame:
    records = []
    for r in results:
        records.append({
            "model_name": r.model_name,
            "degree": r.degree,
            "train_rmse": round(r.train_rmse, 6),
            "test_rmse": round(r.test_rmse, 6),
            "train_r2": round(r.train_r2, 6),
            "test_r2": round(r.test_r2, 6),
        })

    metrics_df = pd.DataFrame(records).sort_values(["model_name", "degree"]).reset_index(drop=True)
    metrics_df.to_csv(outpath, index=False)
    return metrics_df


def print_summary(metrics_df: pd.DataFrame) -> None:
    poly_only = metrics_df[metrics_df["model_name"].isin(["Linear", "Polynomial"])].sort_values("degree")
    best_idx = poly_only["test_rmse"].idxmin()
    best_row = poly_only.loc[best_idx]

    print("\n=== Summary ===")
    print(metrics_df.to_string(index=False))
    print("\nInterpretation:")
    print("- Degree 1 (linear regression) is the baseline model.")
    print(f"- The best model by test RMSE is degree {int(best_row['degree'])}.")
    print("- If training RMSE decreases but test RMSE increases, the model is overfitting.")
    print("- Use the JPG graphs directly in your report or notebook.")


def main() -> None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    raw_df = download_world_bank_indicator(INDICATOR)
    ts = prepare_country_time_series(raw_df, COUNTRY_CODE, COUNTRY_NAME, START_YEAR)
    train_df, test_df = chronological_split(ts, TEST_RATIO)

    clean_csv = OUTPUT_DIR / f"cleaned_{COUNTRY_CODE}.csv"
    ts.to_csv(clean_csv, index=False)

    X_train = train_df[["year"]].to_numpy()
    y_train = train_df["value"].to_numpy()
    X_test = test_df[["year"]].to_numpy()
    y_test = test_df["value"].to_numpy()

    results: List[ModelResult] = []

    for degree in DEGREES:
        model = build_polynomial_model(degree)
        model_name = "Linear" if degree == 1 else "Polynomial"
        result = evaluate_model(model, X_train, y_train, X_test, y_test, degree, model_name)
        results.append(result)

    unregularized_high = next(r for r in results if r.degree == HIGH_DEGREE_FOR_RIDGE)
    ridge_model = build_polynomial_model(HIGH_DEGREE_FOR_RIDGE, ridge_alpha=RIDGE_ALPHA)
    ridge_result = evaluate_model(
        ridge_model,
        X_train,
        y_train,
        X_test,
        y_test,
        HIGH_DEGREE_FOR_RIDGE,
        "Polynomial + Ridge"
    )
    results.append(ridge_result)


    metrics_df = save_metrics(results, OUTPUT_DIR / f"model_metrics_{COUNTRY_CODE}.csv")

    linear_result = next(r for r in results if r.degree == 1)
    poly_candidates = [r for r in results if r.model_name == "Polynomial"]
    best_poly = min(poly_candidates, key=lambda r: r.test_rmse)

    country_label = f"{ts['country_name'].iloc[0]} ({COUNTRY_CODE})"

    plot_raw_series(ts, country_label, OUTPUT_DIR / f"raw_series_{COUNTRY_CODE}.jpg")
    plot_model_fits(ts, train_df, test_df, results, OUTPUT_DIR / f"model_fits_{COUNTRY_CODE}.jpg")
    plot_error_curve(results, OUTPUT_DIR / f"error_vs_degree_{COUNTRY_CODE}.jpg")
    plot_linear_vs_best(
        ts,
        train_df,
        test_df,
        linear_result,
        best_poly,
        OUTPUT_DIR / f"linear_vs_best_polynomial_{COUNTRY_CODE}.jpg"
    )
    plot_regularization_extension(
        ts,
        train_df,
        test_df,
        unregularized_high,
        ridge_result,
        OUTPUT_DIR / f"high_degree_regularization_{COUNTRY_CODE}.jpg"
    )

    print_summary(metrics_df)
    print(f"\nSaved outputs to: {OUTPUT_DIR.resolve()}")


if __name__ == "__main__":
    main()

Reading CSV from archive: API_EN.GHG.CO2.PC.CE.AR5_DS2_en_csv_v2_610.csv

=== Summary ===
        model_name  degree   train_rmse    test_rmse      train_r2       test_r2
            Linear       1 4.079750e-01 2.071759e+00  8.924460e-01 -6.577340e+00
        Polynomial       2 4.049260e-01 1.855356e+00  8.940480e-01 -5.077051e+00
        Polynomial       3 3.291920e-01 6.096640e-01  9.299740e-01  3.438250e-01
        Polynomial       4 3.288860e-01 6.284480e-01  9.301040e-01  3.027680e-01
        Polynomial       5 3.285820e-01 6.479580e-01  9.302340e-01  2.588050e-01
        Polynomial       6 3.282800e-01 6.679490e-01  9.303620e-01  2.123640e-01
        Polynomial       8 3.276840e-01 7.094950e-01  9.306140e-01  1.113380e-01
        Polynomial      10 3.270990e-01 7.529830e-01  9.308620e-01 -9.410000e-04
        Polynomial      12 3.265230e-01 7.982680e-01  9.311050e-01 -1.249580e-01
Polynomial + Ridge      12 5.672434e+09 1.915148e+10 -2.079208e+19 -6.475046e+20

Interpretation:
- 